# 第 11 章：最小二乘法與線性迴歸

本 Notebook 是 `ch11_least_squares.md` 教學文件的精簡互動版，程式碼與 `ch11_least_squares.py` 邏輯一致。

## 學習目標

- 理解超定方程組（overdetermined system）：資料點數量多於未知參數，$Ax=b$ 通常無解
- 理解最小二乘法的目標：最小化殘差平方和 $\|Ax-b\|^2$
- 推導並使用正規方程式 $A^TAx=A^Tb$ 求解
- 使用 QR 分解求解最小二乘（數值上更穩定）
- 將簡單線性迴歸寫成矩陣形式 $Ax=b$，求出最佳擬合直線
- 交叉驗證正規方程式、QR 分解、`np.linalg.lstsq` 三種方法結果一致

In [1]:
import os

import matplotlib
matplotlib.use("Agg")  # Notebook 環境仍以檔案輸出圖片，避免依賴互動式後端

import matplotlib.pyplot as plt
import numpy as np

# 設定中文字型，避免圖片中的中文顯示為方框
matplotlib.rcParams["font.sans-serif"] = [
    "PingFang TC", "PingFang SC", "Heiti TC", "Arial Unicode MS", "DejaVu Sans",
]
matplotlib.rcParams["axes.unicode_minus"] = False

OUT_DIR = os.getcwd()

## 1. 超定方程組 (overdetermined system) 範例

3 個方程式、2 個未知數，一般而言 $Ax=b$ 沒有精確解，只能求最小二乘意義下的最佳近似解。

In [2]:
A_demo = np.array([
    [1.0, 1.0],
    [1.0, 2.0],
    [1.0, 3.0],
])
b_demo = np.array([2.0, 3.0, 5.0])

print("矩陣 A =")
print(A_demo)
print("向量 b =", b_demo)

x_demo, *_ = np.linalg.lstsq(A_demo, b_demo, rcond=None)
print("最小二乘解 x =", x_demo)
print("殘差平方和 ||Ax-b||^2 =", np.sum((A_demo @ x_demo - b_demo) ** 2))

矩陣 A =
[[1. 1.]
 [1. 2.]
 [1. 3.]]
向量 b = [2. 3. 5.]
最小二乘解 x = [0.33333333 1.5       ]
殘差平方和 ||Ax-b||^2 = 0.16666666666666657


## 2. 模擬資料：$y = 2x + 1 + \text{noise}$

用 numpy 產生 30 個帶雜訊的資料點，並固定 random seed 以便結果可重現。

In [3]:
np.random.seed(42)  # 固定隨機種子，確保結果可重現

n_points = 30
x_data = np.linspace(0, 10, n_points)
true_slope = 2.0
true_intercept = 1.0
noise = np.random.normal(loc=0.0, scale=1.5, size=n_points)
y_data = true_slope * x_data + true_intercept + noise

print(f"資料點數量: {n_points}")
print(f"真實斜率 = {true_slope}, 真實截距 = {true_intercept}")
print("前 5 筆 (x, y):")
for i in range(5):
    print(f"  ({x_data[i]:.4f}, {y_data[i]:.4f})")

資料點數量: 30
真實斜率 = 2.0, 真實截距 = 1.0
前 5 筆 (x, y):
  (0.0000, 1.7451)
  (0.3448, 1.4823)
  (0.6897, 3.3508)
  (1.0345, 5.3535)
  (1.3793, 3.4074)


## 3. 矩陣形式 $Ax=b$

$A$ 的欄為 $[x, 1]$，$x=[m,c]^T$（斜率、截距），$b=y$。

In [4]:
A = np.column_stack([x_data, np.ones(n_points)])
b = y_data

print(f"A 的形狀 = {A.shape}, b 的形狀 = {b.shape}")
print("A 的前 5 列 =")
print(A[:5])

A 的形狀 = (30, 2), b 的形狀 = (30,)
A 的前 5 列 =
[[0.         1.        ]
 [0.34482759 1.        ]
 [0.68965517 1.        ]
 [1.03448276 1.        ]
 [1.37931034 1.        ]]


## 4. 方法一：正規方程式 (normal equation)

$$A^TA\,x = A^Tb \quad\Longrightarrow\quad x = (A^TA)^{-1}A^Tb$$

In [5]:
AtA = A.T @ A
Atb = A.T @ b
print("A^T A =")
print(AtA)
print("A^T b =", Atb)

x_normal = np.linalg.inv(AtA) @ Atb
slope_normal, intercept_normal = x_normal
print(f"正規方程式解得：斜率 m = {slope_normal:.6f}, 截距 c = {intercept_normal:.6f}")

A^T A =
[[1017.24137931  150.        ]
 [ 150.           30.        ]]
A^T b = [2101.29983745  321.53338969]
正規方程式解得：斜率 m = 1.847142, 截距 c = 1.482068


## 5. 方法二：QR 分解求解最小二乘

$A=QR$ 代入正規方程式，化簡為 $Rx=Q^Tb$，用回代法求解，數值上比正規方程式更穩定（避免計算 $A^TA$ 造成條件數平方放大）。

In [6]:
Q, R = np.linalg.qr(A)
print(f"Q 的形狀 = {Q.shape}, R 的形狀 = {R.shape}")
print("R (上三角矩陣) =")
print(R)

Qtb = Q.T @ b
x_qr = np.linalg.solve(R, Qtb)
slope_qr, intercept_qr = x_qr
print(f"QR 分解解得：斜率 m = {slope_qr:.6f}, 截距 c = {intercept_qr:.6f}")

Q 的形狀 = (30, 2), R 的形狀 = (2, 2)
R (上三角矩陣) =
[[-31.89422172  -4.70304625]
 [  0.          -2.80737527]]
QR 分解解得：斜率 m = 1.847142, 截距 c = 1.482068


## 6. 方法三：`np.linalg.lstsq`

In [7]:
x_lstsq, residuals, rank, singular_values = np.linalg.lstsq(A, b, rcond=None)
slope_lstsq, intercept_lstsq = x_lstsq
print(f"np.linalg.lstsq 解得：斜率 m = {slope_lstsq:.6f}, 截距 c = {intercept_lstsq:.6f}")
print(f"矩陣 A 的秩 (rank) = {rank}")

np.linalg.lstsq 解得：斜率 m = 1.847142, 截距 c = 1.482068
矩陣 A 的秩 (rank) = 2


## 7. 交叉驗證：三種方法結果應一致

In [8]:
print("正規方程式解 x =", x_normal)
print("QR 分解解     x =", x_qr)
print("lstsq 解      x =", x_lstsq)

match_normal_qr = np.allclose(x_normal, x_qr)
match_normal_lstsq = np.allclose(x_normal, x_lstsq)
match_qr_lstsq = np.allclose(x_qr, x_lstsq)

print("正規方程式 與 QR 分解 是否一致 (np.allclose):", match_normal_qr)
print("正規方程式 與 lstsq   是否一致 (np.allclose):", match_normal_lstsq)
print("QR 分解     與 lstsq   是否一致 (np.allclose):", match_qr_lstsq)

assert match_normal_qr and match_normal_lstsq and match_qr_lstsq, "三種方法結果不一致！"
print("驗證通過：三種方法求出的最小二乘解完全一致。")

print()
print(f"最終擬合直線: y = {slope_lstsq:.4f} * x + {intercept_lstsq:.4f}")
print(f"（真實直線為 y = {true_slope} * x + {true_intercept}）")

正規方程式解 x = [1.84714242 1.48206754]
QR 分解解     x = [1.84714242 1.48206754]
lstsq 解      x = [1.84714242 1.48206754]
正規方程式 與 QR 分解 是否一致 (np.allclose): True
正規方程式 與 lstsq   是否一致 (np.allclose): True
QR 分解     與 lstsq   是否一致 (np.allclose): True
驗證通過：三種方法求出的最小二乘解完全一致。

最終擬合直線: y = 1.8471 * x + 1.4821
（真實直線為 y = 2.0 * x + 1.0）


## 8. 繪圖：資料點與最佳擬合直線

In [9]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(x_data, y_data, color="tab:blue", alpha=0.7, label="模擬資料點 (帶雜訊)")

x_line = np.linspace(x_data.min(), x_data.max(), 100)
y_fit = slope_lstsq * x_line + intercept_lstsq
y_true = true_slope * x_line + true_intercept

ax.plot(x_line, y_fit, color="tab:red", linewidth=2,
        label=f"最小二乘擬合: y = {slope_lstsq:.3f}x + {intercept_lstsq:.3f}")
ax.plot(x_line, y_true, color="tab:green", linewidth=1.5, linestyle="--",
        label=f"真實直線: y = {true_slope}x + {true_intercept}")

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("最小二乘法線性迴歸：資料點與最佳擬合直線")
ax.legend(loc="upper left")
ax.grid(True, linestyle="--", alpha=0.5)

fig_path = os.path.join(OUT_DIR, "regression_fit.png")
plt.savefig(fig_path, dpi=120, bbox_inches="tight")
plt.show()
print("已儲存資料擬合圖至:", fig_path)

已儲存資料擬合圖至: /Users/rexwang/workspace/linear-algebra-with-matlab-python-tutorial/ch11_least_squares/regression_fit.png


/var/folders/3r/mbt455ns6n3fn8ythcyc95qc0000gn/T/ipykernel_51987/1098524429.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 重點整理

- **超定方程組**：方程式數量多於未知數，$Ax=b$ 通常無精確解，需求最小二乘近似解。
- **正規方程式** $A^TAx=A^Tb$：由梯度為零或殘差正交於行空間推導而來。
- **QR 分解** $Rx=Q^Tb$：數值上比正規方程式更穩定，避免條件數平方放大。
- **簡單線性迴歸**：把 $y=mx+c$ 寫成矩陣形式 $Ax=b$（$A$ 的欄為 $[x,1]$），用最小二乘法求解。
- 三種解法（正規方程式、QR 分解、`np.linalg.lstsq`）在本例中完全一致。